# Causal LM & Instruction Tuning

> GPT-style fine-tuning with optional instruction formatting and PEFT/LoRA adapters.

This notebook starts with plain next-token modeling and then moves to instruction tuning and adapter workflows.

## Notebook Metadata

`nbdev` export target for this notebook.

In [ ]:
#| default_exp text.causal_lm


## Setup

Import dependencies.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import pandas as pd

from blurr.text.causal_lm import CausalLMLearner


## Quick Start

Fine-tune a tiny GPT-2 checkpoint on raw text and generate completions.

In [ ]:
lm_df = pd.DataFrame({
    'text': [
        'The quick brown fox jumps over the lazy dog.',
        'Machine learning models improve with high-quality data.',
        'Open-source communities accelerate innovation.',
        'Careful evaluation is essential before deployment.',
    ]
})

learn = CausalLMLearner.from_df(
    lm_df,
    model_name='sshleifer/tiny-gpt2',
    text_col='text',
    max_length=96,
    bs=2,
)

learn.fit_one_cycle(1, 2e-4)
quick_completion = learn.blurr_generate('Model evaluation', max_new_tokens=40)[0]
quick_completion


**Expected output**

A dictionary with `prompt`, `completion`, and `generated_text`.
`completion` is only the new continuation tokens.

## Instruction Tuning

Provide `instruction_col` + `response_col` to automatically format supervised instruction data.
With `mask_prompt=True`, loss is applied to response tokens only.

In [ ]:
instruct_df = pd.DataFrame({
    'instruction': [
        'Summarize this sentence: Open-source tooling speeds up experimentation.',
        'Translate to French: Good morning, how are you?',
        'Answer briefly: What is the capital of Japan?',
    ],
    'response': [
        'Open-source tools help teams experiment faster.',
        'Bonjour, comment allez-vous ?',
        'Tokyo.',
    ],
})

learn_instruct = CausalLMLearner.from_df(
    instruct_df,
    model_name='sshleifer/tiny-gpt2',
    instruction_col='instruction',
    response_col='response',
    mask_prompt=True,
    max_length=128,
    bs=2,
)

learn_instruct.fit_one_cycle(1, 2e-4)


## Advanced: Custom Prompt Templates

Use `prompt_template` when matching an existing instruction/chat format.

In [ ]:
learn_custom = CausalLMLearner.from_df(
    instruct_df,
    model_name='sshleifer/tiny-gpt2',
    instruction_col='instruction',
    response_col='response',
    prompt_template='<|user|>\n{instruction}\n<|assistant|>\n{response}',
    max_length=128,
    bs=2,
)


## Advanced: PEFT / LoRA

For larger models, train adapters instead of full model weights.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['c_attn'],
    lora_dropout=0.1,
)

learn_peft = CausalLMLearner.from_df(
    lm_df,
    model_name='sshleifer/tiny-gpt2',
    text_col='text',
    peft_config=peft_config,
    max_length=96,
    bs=2,
)


### Save & Load Adapters

These methods are available for PEFT-enabled learners.

In [ ]:
# Save adapter weights
# learn_peft.save_peft('./my_lora_adapter')

# Load adapter on a base model
# model = CausalLMLearner.load_peft('sshleifer/tiny-gpt2', './my_lora_adapter')


## API Reference

### Main class
- `CausalLMLearner`

### Construction methods
- `from_df(df, model_name, ...)`
- `from_hf(dataset, subset=None, model_name=..., ...)`

### Inference
- `blurr_generate(texts, max_new_tokens=..., temperature=..., top_p=..., ...)`

### PEFT helpers
- `save_peft(path)`
- `load_peft(base_model_name, adapter_path)`

### Utilities
- `format_instruction_data(...)` (via `blurr.text.causal_lm`) for preparing supervised instruction data.
